# Phase 0 Pipeline Acceptance Notebook

Human-readable validation for the Phase 0 BTC data pipeline.

This notebook is meant to answer four acceptance questions:

1. Did the canonical OHLCV parquet load correctly with the expected schema and UTC hourly timestamps?
2. Did reconcile/validation artifacts exist and remain internally consistent?
3. Are known data anomalies documented rather than silently hidden?
4. Do market-event spot checks and source handoff checks look sane enough for Phase 1 backtesting?

Notes:

- The optional pipeline rebuild cell is disabled by default so `Run All` does not mutate data.
- Missing bars and zero-volume bars are not automatic failures in Phase 0; they are documented acceptance artifacts.
- The final cell summarizes the acceptance verdict.

## 0. Setup

In [1]:
from pathlib import Path
import glob
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display



def _bootstrap_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "backtest").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            os.chdir(candidate)
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


PROJECT_ROOT = _bootstrap_repo_root()
print("Notebook repo root:", PROJECT_ROOT)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

CANONICAL_PATH = Path("data/raw/btcusdt_1h.parquet")
QUALITY_DIR = Path("data/quality")
ARCHIVE_DIR = Path("data/raw/archive")
UPDATE_PATH = Path("data/raw/btcusdt_1h_update.parquet")

PHASE0_ACCEPTANCE = {}


def pass_fail(condition):
    return "PASS" if bool(condition) else "FAIL"


def check_row(check, passed, detail=""):
    return {"check": check, "status": pass_fail(passed), "detail": detail}


def display_checks(rows, title=None, require_all=True):
    df_checks = pd.DataFrame(rows)
    if title:
        print(title)
    display(df_checks)
    if require_all:
        failed = df_checks[df_checks["status"] != "PASS"]
        assert failed.empty, failed
    return df_checks


## 1. Optional Pipeline Rebuild

This cell is intentionally guarded. Set `RUN_PIPELINE = True` only when you want to refresh data by running the Phase 0 ingestion flow:

1. bulk Binance Vision load
2. incremental CCXT update
3. reconcile into canonical parquet and archive prior snapshot

For ordinary acceptance review, leave it disabled and validate the current artifacts.

In [2]:
RUN_PIPELINE = False

pipeline_commands = [
    [sys.executable, "-m", "ingestion.bulk_download", "--pair", "BTCUSDT", "--interval", "1h", "--start", "2020-01"],
    [sys.executable, "-m", "ingestion.incremental_update", "--pair", "BTCUSDT", "--interval", "1h"],
    [sys.executable, "-m", "ingestion.reconcile", "--existing", str(CANONICAL_PATH), "--new", str(UPDATE_PATH)],
]

if RUN_PIPELINE:
    for cmd in pipeline_commands:
        print("Running:", " ".join(cmd))
        completed = subprocess.run(cmd, capture_output=True, text=True)
        print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        assert completed.returncode == 0, f"Pipeline command failed: {' '.join(cmd)}"
    print("Pipeline execution completed.")
else:
    print("Pipeline rebuild skipped. Set RUN_PIPELINE = True to refresh data.")


Pipeline rebuild skipped. Set RUN_PIPELINE = True to refresh data.


## 2. Load Canonical Dataset

Load the canonical Phase 0 parquet and show the headline facts: row count, date range, source coverage, and schema.

In [3]:
assert CANONICAL_PATH.exists(), f"Canonical parquet not found: {CANONICAL_PATH}"

df = pd.read_parquet(CANONICAL_PATH).sort_values("open_time_utc").reset_index(drop=True)

canonical_summary = pd.DataFrame([
    {"field": "canonical_path", "value": str(CANONICAL_PATH)},
    {"field": "rows", "value": len(df)},
    {"field": "start_utc", "value": df["open_time_utc"].min()},
    {"field": "end_utc", "value": df["open_time_utc"].max()},
    {"field": "columns", "value": ", ".join(df.columns)},
])

display(canonical_summary)
display(df.head(3))
display(df.tail(3))

PHASE0_ACCEPTANCE["canonical_load"] = True


,field,value
0,canonical_path,data/raw/btcusdt_1h.parquet
1,rows,55105
2,start_utc,2020-01-01 00:00:00+00:00
3,end_utc,2026-04-16 07:00:00+00:00
4,columns,"open_time_utc, open, high, low, close, volume, quote_volume, trade_count, source, ingested_at_utc"


,open_time_utc,open,high,low,close,volume,quote_volume,trade_count,source,ingested_at_utc
0,2020-01-01 00:00:00+00:00,7195.24,7196.25,7175.46,7177.02,511.814901,3.675857e+06,7640,binance_vision,2026-04-16 08:14:07.959000+00:00
1,2020-01-01 01:00:00+00:00,7176.47,7230.00,7175.71,7216.27,883.052603,6.365953e+06,9033,binance_vision,2026-04-16 08:14:07.959000+00:00
2,2020-01-01 02:00:00+00:00,7215.52,7244.87,7211.41,7242.85,655.156809,4.736719e+06,7466,binance_vision,2026-04-16 08:14:07.959000+00:00


,open_time_utc,open,high,low,close,volume,quote_volume,trade_count,source,ingested_at_utc
55102,2026-04-16 05:00:00+00:00,74944.15,75139.82,74900.00,75043.48,359.01217,2.694262e+07,84505,ccxt_binance,2026-04-16 08:14:14.334000+00:00
55103,2026-04-16 06:00:00+00:00,75043.47,75115.12,74853.72,75045.01,604.84284,4.537296e+07,82824,ccxt_binance,2026-04-16 08:14:14.334000+00:00
55104,2026-04-16 07:00:00+00:00,75045.01,75082.78,74636.00,74707.03,708.06090,5.301598e+07,91556,ccxt_binance,2026-04-16 08:14:14.334000+00:00


## 3. Core Schema And Timestamp Checks

These are hard acceptance checks. Phase 1 backtests assume UTC-aware, sorted, unique, hour-aligned OHLCV rows.

In [4]:
required_columns = [
    "open_time_utc",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_volume",
    "trade_count",
    "source",
    "ingested_at_utc",
]
missing_columns = sorted(set(required_columns) - set(df.columns))

timezone_is_utc = str(df["open_time_utc"].dt.tz) == "UTC"
non_aligned_rows = df[
    (df["open_time_utc"].dt.minute != 0) |
    (df["open_time_utc"].dt.second != 0) |
    (df["open_time_utc"].dt.microsecond != 0)
]
duplicate_count = int(df["open_time_utc"].duplicated().sum())
monotonic = bool(df["open_time_utc"].is_monotonic_increasing)
null_counts = df[required_columns].isna().sum().to_dict() if not missing_columns else {}

schema_checks = [
    check_row("required columns present", not missing_columns, f"missing={missing_columns}"),
    check_row("open_time_utc is timezone-aware UTC", timezone_is_utc, str(df["open_time_utc"].dt.tz)),
    check_row("timestamps sorted ascending", monotonic),
    check_row("timestamps unique", duplicate_count == 0, f"duplicates={duplicate_count}"),
    check_row("timestamps exactly hourly aligned", len(non_aligned_rows) == 0, f"unaligned={len(non_aligned_rows)}"),
    check_row("OHLCV required fields non-null", all(v == 0 for v in null_counts.values()), str(null_counts)),
    check_row("price sanity: high >= open/close/low", bool(((df["high"] >= df[["open", "close", "low"]].max(axis=1))).all())),
    check_row("price sanity: low <= open/close/high", bool(((df["low"] <= df[["open", "close", "high"]].min(axis=1))).all())),
]

display_checks(schema_checks, "Core schema/timestamp checks")
PHASE0_ACCEPTANCE["schema_timestamp"] = True


Core schema/timestamp checks


,check,status,detail
0,required columns present,PASS,missing=[]
1,open_time_utc is timezone-aware UTC,PASS,UTC
2,timestamps sorted ascending,PASS,
3,timestamps unique,PASS,duplicates=0
4,timestamps exactly hourly aligned,PASS,unaligned=0
5,OHLCV required fields non-null,PASS,"{'open_time_utc': 0, 'open': 0, 'high': 0, 'low': 0, 'close': 0, 'volume': 0, 'quote_volume': 0, 'trade_count': 0, '..."
6,price sanity: high >= open/close/low,PASS,
7,price sanity: low <= open/close/high,PASS,


## 4. Source Coverage And Reconcile Handoff

This section checks how much data came from each source and whether the transition from historical bulk data to incremental API data is clean.

In [5]:
source_counts = df["source"].value_counts(dropna=False).rename_axis("source").reset_index(name="rows")
source_coverage = (
    df.groupby("source", dropna=False)
    .agg(
        rows=("open_time_utc", "size"),
        start_utc=("open_time_utc", "min"),
        end_utc=("open_time_utc", "max"),
    )
    .reset_index()
)

display(source_counts)
display(source_coverage)

handoff_rows = []
if {"binance_vision", "ccxt_binance"}.issubset(set(df["source"].dropna().unique())):
    bv_last = df.loc[df["source"] == "binance_vision", "open_time_utc"].max()
    ccxt_first = df.loc[df["source"] == "ccxt_binance", "open_time_utc"].min()
    gap_hours = (ccxt_first - bv_last).total_seconds() / 3600
    bv_last_close = float(df.loc[df["open_time_utc"] == bv_last, "close"].iloc[0])
    ccxt_first_open = float(df.loc[df["open_time_utc"] == ccxt_first, "open"].iloc[0])
    pct_diff = abs(ccxt_first_open / bv_last_close - 1) * 100

    handoff_rows = [
        check_row("source handoff exactly one hour", gap_hours == 1, f"gap_hours={gap_hours:.0f}"),
        check_row("source handoff price continuity < 1%", pct_diff < 1.0, f"pct_diff={pct_diff:.4f}%"),
    ]

    boundary = df[
        (df["open_time_utc"] >= bv_last - pd.Timedelta(hours=2)) &
        (df["open_time_utc"] <= ccxt_first + pd.Timedelta(hours=2))
    ][["open_time_utc", "open", "high", "low", "close", "volume", "source"]]
    display(boundary)
else:
    handoff_rows = [check_row("source handoff check applicable", True, "ccxt_binance or binance_vision missing; no handoff expected yet")]

display_checks(handoff_rows, "Source handoff checks")
PHASE0_ACCEPTANCE["source_handoff"] = True


,source,rows
0,binance_vision,54737
1,ccxt_binance,368


,source,rows,start_utc,end_utc
0,binance_vision,54737,2020-01-01 00:00:00+00:00,2026-03-31 23:00:00+00:00
1,ccxt_binance,368,2026-04-01 00:00:00+00:00,2026-04-16 07:00:00+00:00


,open_time_utc,open,high,low,close,volume,source
54734,2026-03-31 21:00:00+00:00,68265.53,68410.30,67884.00,68277.99,494.48096,binance_vision
54735,2026-03-31 22:00:00+00:00,68278.00,68341.16,67962.27,68172.87,755.55219,binance_vision
54736,2026-03-31 23:00:00+00:00,68173.59,68371.10,68090.20,68284.48,610.99275,binance_vision
54737,2026-04-01 00:00:00+00:00,68284.49,68343.77,67853.30,68330.75,542.79511,ccxt_binance
54738,2026-04-01 01:00:00+00:00,68330.76,68378.64,67760.33,67782.73,966.68117,ccxt_binance
54739,2026-04-01 02:00:00+00:00,67782.74,67913.98,67578.75,67668.89,1114.17912,ccxt_binance


Source handoff checks


,check,status,detail
0,source handoff exactly one hour,PASS,gap_hours=1
1,source handoff price continuity < 1%,PASS,pct_diff=0.0000%


## 5. Archive Snapshot Integrity

Reconcile should preserve prior canonical snapshots under `data/raw/archive/`. Archive presence is especially important once incremental rows have been merged.

In [6]:
archives = sorted(ARCHIVE_DIR.glob("*.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
has_incremental_source = "ccxt_binance" in set(df["source"].dropna().unique()) or "ccxt_api" in set(df["source"].dropna().unique())

archive_rows = []
if not archives and not has_incremental_source:
    archive_rows.append(check_row("archive optional before incremental reconcile", True, "no incremental source found"))
elif not archives and has_incremental_source:
    archive_rows.append(check_row("archive exists after incremental reconcile", False, "incremental source exists but archive directory has no parquet snapshots"))
else:
    latest_archive = archives[0]
    archive_df = pd.read_parquet(latest_archive)
    row_diff = len(df) - len(archive_df)
    archive_rows.extend([
        check_row("archive exists", True, latest_archive.name),
        check_row("archive filename has UTC timestamp markers", "T" in latest_archive.name and "Z" in latest_archive.name, latest_archive.name),
        check_row("canonical rows >= latest archive rows", row_diff >= 0, f"canonical={len(df)}, archive={len(archive_df)}, diff={row_diff}"),
    ])
    display(pd.DataFrame([
        {"field": "latest_archive", "value": latest_archive.name},
        {"field": "archive_rows", "value": len(archive_df)},
        {"field": "canonical_rows", "value": len(df)},
        {"field": "row_diff", "value": row_diff},
    ]))

display_checks(archive_rows, "Archive integrity checks")
PHASE0_ACCEPTANCE["archive_integrity"] = True


,field,value
0,latest_archive,btcusdt_1h_20260416T081415Z.parquet
1,archive_rows,54737
2,canonical_rows,55105
3,row_diff,368


Archive integrity checks


,check,status,detail
0,archive exists,PASS,btcusdt_1h_20260416T081415Z.parquet
1,archive filename has UTC timestamp markers,PASS,btcusdt_1h_20260416T081415Z.parquet
2,canonical rows >= latest archive rows,PASS,"canonical=55105, archive=54737, diff=368"


## 6. Validation Report Audit

This reads the newest validation report and surfaces the key warning counts that must be understood before backtesting.

In [7]:
reports = sorted(QUALITY_DIR.glob("*validation*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
assert reports, f"No validation report JSON found in {QUALITY_DIR}"

latest_report = reports[0]
report_data = json.loads(latest_report.read_text())
checks = report_data.get("checks", {})

report_summary = pd.DataFrame([
    {"field": "latest_report", "value": latest_report.name},
    {"field": "overall_status", "value": report_data.get("overall_status", "UNKNOWN")},
    {"field": "zero_volume_bars", "value": checks.get("zero_volume_bars", {}).get("count", 0)},
    {"field": "volume_anomaly_count", "value": checks.get("volume_anomaly", {}).get("count", 0)},
    {"field": "price_sanity_count", "value": checks.get("price_sanity", {}).get("count", 0)},
])
display(report_summary)

report_checks = [
    check_row("validation report exists", True, latest_report.name),
    check_row("validator did not report price sanity failures", checks.get("price_sanity", {}).get("count", 0) == 0, str(checks.get("price_sanity", {}))),
]
display_checks(report_checks, "Validation report checks")
PHASE0_ACCEPTANCE["validation_report"] = True


,field,value
0,latest_report,reconcile_validation_20260416.json
1,overall_status,WARNING
2,zero_volume_bars,3
3,volume_anomaly_count,0
4,price_sanity_count,0


Validation report checks


,check,status,detail
0,validation report exists,PASS,reconcile_validation_20260416.json
1,validator did not report price sanity failures,PASS,{}


## 7. Missing Bar Analysis

Phase 0 does not forward-fill or interpolate gaps. Acceptance requires gaps to be explicit, grouped into intervals, and documented.

In [8]:
start_time = df["open_time_utc"].min()
end_time = df["open_time_utc"].max()
expected_index = pd.date_range(start=start_time, end=end_time, freq="1h", tz="UTC")
actual_index = pd.DatetimeIndex(df["open_time_utc"])
missing_index = expected_index.difference(actual_index)

missing_summary = pd.DataFrame([
    {"field": "start_utc", "value": start_time},
    {"field": "end_utc", "value": end_time},
    {"field": "expected_hourly_rows", "value": len(expected_index)},
    {"field": "actual_rows", "value": len(df)},
    {"field": "missing_rows", "value": len(missing_index)},
])
display(missing_summary)

if len(missing_index) > 0:
    missing_series = pd.Series(missing_index, name="missing_time_utc")
    group_id = (missing_series.diff() > pd.Timedelta(hours=1)).cumsum()
    gap_intervals = (
        missing_series.groupby(group_id)
        .agg(["min", "max", "count"])
        .rename(columns={"min": "gap_start_utc", "max": "gap_end_utc", "count": "missing_hours"})
        .reset_index(drop=True)
    )
    gap_intervals["year"] = gap_intervals["gap_start_utc"].dt.year
    display(gap_intervals)
    display(pd.DataFrame({"first_20_missing_time_utc": missing_index[:20]}))
else:
    gap_intervals = pd.DataFrame(columns=["gap_start_utc", "gap_end_utc", "missing_hours", "year"])
    print("No missing hourly bars found.")

missing_checks = [
    check_row("timestamps have no duplicates", df["open_time_utc"].duplicated().sum() == 0),
    check_row("gaps are documented", True, f"missing_rows={len(missing_index)}, intervals={len(gap_intervals)}"),
    check_row("no missing bars in 2024+", not any(gap_intervals["year"] >= 2024) if len(gap_intervals) else True),
]
display_checks(missing_checks, "Missing-bar acceptance checks")
PHASE0_ACCEPTANCE["missing_bar_documentation"] = True


,field,value
0,start_utc,2020-01-01 00:00:00+00:00
1,end_utc,2026-04-16 07:00:00+00:00
2,expected_hourly_rows,55136
3,actual_rows,55105
4,missing_rows,31


,gap_start_utc,gap_end_utc,missing_hours,year
0,2020-02-09 02:00:00+00:00,2020-02-09 02:00:00+00:00,1,2020
1,2020-02-19 12:00:00+00:00,2020-02-19 16:00:00+00:00,5,2020
2,2020-03-04 10:00:00+00:00,2020-03-04 10:00:00+00:00,1,2020
3,2020-04-25 02:00:00+00:00,2020-04-25 03:00:00+00:00,2,2020
4,2020-06-28 02:00:00+00:00,2020-06-28 04:00:00+00:00,3,2020
5,2020-11-30 06:00:00+00:00,2020-11-30 06:00:00+00:00,1,2020
6,2020-12-21 15:00:00+00:00,2020-12-21 17:00:00+00:00,3,2020
7,2020-12-25 02:00:00+00:00,2020-12-25 02:00:00+00:00,1,2020
8,2021-02-11 04:00:00+00:00,2021-02-11 04:00:00+00:00,1,2021
9,2021-03-06 02:00:00+00:00,2021-03-06 02:00:00+00:00,1,2021


,first_20_missing_time_utc
0,2020-02-09 02:00:00+00:00
1,2020-02-19 12:00:00+00:00
2,2020-02-19 13:00:00+00:00
3,2020-02-19 14:00:00+00:00
4,2020-02-19 15:00:00+00:00
5,2020-02-19 16:00:00+00:00
6,2020-03-04 10:00:00+00:00
7,2020-04-25 02:00:00+00:00
8,2020-04-25 03:00:00+00:00
9,2020-06-28 02:00:00+00:00


Missing-bar acceptance checks


,check,status,detail
0,timestamps have no duplicates,PASS,
1,gaps are documented,PASS,"missing_rows=31, intervals=15"
2,no missing bars in 2024+,PASS,


### 7a. Gap Context

Show nearby real rows around each missing interval. This makes the anomaly review concrete without altering the canonical dataset.

In [9]:
context_hours = 3
gap_context_frames = []

for i, row in gap_intervals.iterrows():
    gap_start = row["gap_start_utc"]
    gap_end = row["gap_end_utc"]
    context = df[
        (df["open_time_utc"] >= gap_start - pd.Timedelta(hours=context_hours)) &
        (df["open_time_utc"] <= gap_end + pd.Timedelta(hours=context_hours))
    ][["open_time_utc", "open", "high", "low", "close", "volume", "source"]].copy()
    context.insert(0, "gap_id", i + 1)
    context.insert(1, "gap_window", f"{gap_start} -> {gap_end}")
    gap_context_frames.append(context)

if gap_context_frames:
    gap_context = pd.concat(gap_context_frames, ignore_index=True)
    display(gap_context)
else:
    print("No gap context needed; no gaps found.")


,gap_id,gap_window,open_time_utc,open,high,low,close,volume,source
0,1,2020-02-09 02:00:00+00:00 -> 2020-02-09 02:00:00+00:00,2020-02-08 23:00:00+00:00,9913.26,9923.00,9892.50,9895.05,832.750529,binance_vision
1,1,2020-02-09 02:00:00+00:00 -> 2020-02-09 02:00:00+00:00,2020-02-09 00:00:00+00:00,9895.04,9947.50,9880.75,9936.95,1430.944116,binance_vision
2,1,2020-02-09 02:00:00+00:00 -> 2020-02-09 02:00:00+00:00,2020-02-09 01:00:00+00:00,9937.05,9944.25,9909.56,9938.90,957.502674,binance_vision
3,1,2020-02-09 02:00:00+00:00 -> 2020-02-09 02:00:00+00:00,2020-02-09 03:00:00+00:00,9938.86,10052.67,9937.92,10052.63,4894.060253,binance_vision
4,1,2020-02-09 02:00:00+00:00 -> 2020-02-09 02:00:00+00:00,2020-02-09 04:00:00+00:00,10052.66,10131.00,10052.22,10091.27,3942.411537,binance_vision
...,...,...,...,...,...,...,...,...,...
85,15,2023-03-24 13:00:00+00:00 -> 2023-03-24 13:00:00+00:00,2023-03-24 11:00:00+00:00,28039.71,28091.03,27963.84,28080.00,1267.417140,binance_vision
86,15,2023-03-24 13:00:00+00:00 -> 2023-03-24 13:00:00+00:00,2023-03-24 12:00:00+00:00,28080.00,28080.00,28080.00,28080.00,0.000000,binance_vision
87,15,2023-03-24 13:00:00+00:00 -> 2023-03-24 13:00:00+00:00,2023-03-24 14:00:00+00:00,28079.99,28253.01,27835.00,27989.06,8983.240180,binance_vision
88,15,2023-03-24 13:00:00+00:00 -> 2023-03-24 13:00:00+00:00,2023-03-24 15:00:00+00:00,27989.07,28076.82,27843.41,28018.04,5198.286810,binance_vision


## 8. Zero-Volume Bars

Zero-volume bars are retained and flagged. For Phase 1 execution, fills on zero-volume bars are deferred by the execution model.

In [10]:
zero_vol = df[df["volume"] == 0][["open_time_utc", "open", "high", "low", "close", "volume", "source"]].copy()

if len(missing_index) > 0 and not zero_vol.empty:
    proximity_hours = 6
    gap_hours_set = set(missing_index)

    def near_gap(ts):
        return any(ts + pd.Timedelta(hours=h) in gap_hours_set for h in range(-proximity_hours, proximity_hours + 1))

    zero_vol["within_6h_of_missing_gap"] = zero_vol["open_time_utc"].apply(near_gap)

zero_summary = pd.DataFrame([
    {"field": "zero_volume_rows", "value": len(zero_vol)},
    {"field": "within_6h_of_gap", "value": int(zero_vol.get("within_6h_of_missing_gap", pd.Series(dtype=bool)).sum()) if not zero_vol.empty else 0},
])
display(zero_summary)
if zero_vol.empty:
    print("No zero-volume bars found.")
else:
    display(zero_vol)

zero_checks = [
    check_row("zero-volume rows documented", True, f"zero_volume_rows={len(zero_vol)}"),
    check_row("zero-volume rows are retained, not silently dropped", len(zero_vol) >= 0),
]
display_checks(zero_checks, "Zero-volume documentation checks")
PHASE0_ACCEPTANCE["zero_volume_documentation"] = True


,field,value
0,zero_volume_rows,3
1,within_6h_of_gap,3


,open_time_utc,open,high,low,close,volume,source,within_6h_of_missing_gap
8521,2020-12-21 14:00:00+00:00,22646.53,22646.53,22646.53,22646.53,0.0,binance_vision,True
9754,2021-02-11 03:00:00+00:00,44582.07,44582.07,44582.07,44582.07,0.0,binance_vision,True
28254,2023-03-24 12:00:00+00:00,28080.00,28080.00,28080.00,28080.00,0.0,binance_vision,True


Zero-volume documentation checks


,check,status,detail
0,zero-volume rows documented,PASS,zero_volume_rows=3
1,"zero-volume rows are retained, not silently dropped",PASS,


## 9. Missing Rows By Month

In [11]:
if len(missing_index) > 0:
    missing_by_month = (
        pd.Series(1, index=missing_index)
        .resample("ME")
        .sum()
        .fillna(0)
        .astype(int)
        .rename("missing_rows")
    )
    missing_by_month = missing_by_month[missing_by_month > 0]
    display(missing_by_month.reset_index().rename(columns={"index": "month_end_utc"}))
else:
    print("No missing rows by month.")


,month_end_utc,missing_rows
0,2020-02-29 00:00:00+00:00,6
1,2020-03-31 00:00:00+00:00,1
2,2020-04-30 00:00:00+00:00,2
3,2020-06-30 00:00:00+00:00,3
4,2020-11-30 00:00:00+00:00,1
5,2020-12-31 00:00:00+00:00,4
6,2021-02-28 00:00:00+00:00,1
7,2021-03-31 00:00:00+00:00,1
8,2021-04-30 00:00:00+00:00,5
9,2021-08-31 00:00:00+00:00,4


## 10. Known Market Event Spot Checks

These checks make sure the price history roughly matches well-known BTC market events. They are not a replacement for raw data validation, but they catch obvious symbol/source mistakes.

In [12]:
known_events = [
    {"name": "COVID Crash - Black Thursday", "date": "2020-03-12", "check": "daily_low", "expected_range": (3800, 5500), "notes": "BTC crashed from around 7900 to around 3800 intraday"},
    {"name": "COVID Crash - Recovery Start", "date": "2020-03-13", "check": "daily_close_approx", "expected_range": (5000, 6000), "notes": "Bounce day after crash"},
    {"name": "2020 Halving Day", "date": "2020-05-11", "check": "daily_close_approx", "expected_range": (8500, 9200), "notes": "Third BTC halving"},
    {"name": "BTC First Hits 20K", "date": "2020-12-16", "check": "daily_high", "expected_range": (20000, 22000), "notes": "First break above 2017 ATH"},
    {"name": "BTC ATH Window - Nov 2021", "date": "2021-11-10", "check": "daily_high", "expected_range": (66000, 69500), "notes": "ATH window around 69K"},
    {"name": "LUNA/UST Collapse", "date": "2022-05-12", "check": "daily_low", "expected_range": (26000, 30000), "notes": "Terra collapse drawdown"},
    {"name": "FTX Collapse - Nov 2022", "date": "2022-11-09", "check": "daily_low", "expected_range": (15500, 18000), "notes": "FTX bankruptcy drawdown"},
    {"name": "Cycle Low - Post FTX", "date": "2022-11-21", "check": "daily_low", "expected_range": (15400, 16500), "notes": "Approximate cycle bottom"},
    {"name": "BTC ETF Approval Day", "date": "2024-01-11", "check": "daily_high", "expected_range": (46000, 49500), "notes": "Spot BTC ETF approval window"},
    {"name": "2024 Halving Day", "date": "2024-04-20", "check": "daily_close_approx", "expected_range": (63000, 66000), "notes": "Fourth BTC halving"},
]

spot_rows = []
for event in known_events:
    date = pd.Timestamp(event["date"], tz="UTC")
    day_data = df[df["open_time_utc"].dt.date == date.date()]
    if day_data.empty:
        actual = np.nan
        status = "FAIL"
    elif event["check"] == "daily_low":
        actual = float(day_data["low"].min())
        status = pass_fail(event["expected_range"][0] <= actual <= event["expected_range"][1])
    elif event["check"] == "daily_high":
        actual = float(day_data["high"].max())
        status = pass_fail(event["expected_range"][0] <= actual <= event["expected_range"][1])
    else:
        actual = float(day_data.iloc[-1]["close"])
        status = pass_fail(event["expected_range"][0] <= actual <= event["expected_range"][1])

    spot_rows.append({
        "event": event["name"],
        "date": event["date"],
        "check": event["check"],
        "expected_low": event["expected_range"][0],
        "expected_high": event["expected_range"][1],
        "actual": actual,
        "status": status,
        "notes": event["notes"],
    })

spot_df = pd.DataFrame(spot_rows)
display(spot_df)

failed_spot = spot_df[spot_df["status"] != "PASS"]
assert failed_spot.empty, failed_spot
PHASE0_ACCEPTANCE["market_event_spot_checks"] = True
print(f"PASS: {len(spot_df)}/{len(spot_df)} known market events are within expected ranges.")


,event,date,check,expected_low,expected_high,actual,status,notes
0,COVID Crash - Black Thursday,2020-03-12,daily_low,3800,5500,4410.00,PASS,BTC crashed from around 7900 to around 3800 intraday
1,COVID Crash - Recovery Start,2020-03-13,daily_close_approx,5000,6000,5578.60,PASS,Bounce day after crash
2,2020 Halving Day,2020-05-11,daily_close_approx,8500,9200,8561.52,PASS,Third BTC halving
3,BTC First Hits 20K,2020-12-16,daily_high,20000,22000,21560.00,PASS,First break above 2017 ATH
4,BTC ATH Window - Nov 2021,2021-11-10,daily_high,66000,69500,69000.00,PASS,ATH window around 69K
5,LUNA/UST Collapse,2022-05-12,daily_low,26000,30000,26700.00,PASS,Terra collapse drawdown
6,FTX Collapse - Nov 2022,2022-11-09,daily_low,15500,18000,15588.00,PASS,FTX bankruptcy drawdown
7,Cycle Low - Post FTX,2022-11-21,daily_low,15400,16500,15476.00,PASS,Approximate cycle bottom
8,BTC ETF Approval Day,2024-01-11,daily_high,46000,49500,48969.48,PASS,Spot BTC ETF approval window
9,2024 Halving Day,2024-04-20,daily_close_approx,63000,66000,64940.59,PASS,Fourth BTC halving


PASS: 10/10 known market events are within expected ranges.


## 11. Statistical Sanity Checks

The yearly trajectory should look broadly like BTC history. This section summarizes annual prices and volumes so odd source or scale mistakes are visible.

In [13]:
df_with_year = df.copy()
df_with_year["year"] = df_with_year["open_time_utc"].dt.year

yearly = df_with_year.groupby("year").agg(
    rows=("close", "size"),
    open_price=("open", "first"),
    close_price=("close", "last"),
    year_high=("high", "max"),
    year_low=("low", "min"),
    avg_hourly_volume=("volume", "mean"),
).round(2)
yearly["year_return_pct"] = ((yearly["close_price"] / yearly["open_price"] - 1) * 100).round(1)

display(yearly)

stat_checks = [
    check_row("all yearly highs exceed yearly lows", bool((yearly["year_high"] > yearly["year_low"]).all())),
    check_row("all yearly row counts positive", bool((yearly["rows"] > 0).all())),
    check_row("prices are positive", bool((df[["open", "high", "low", "close"]] > 0).all().all())),
    check_row("volume is non-negative", bool((df["volume"] >= 0).all())),
]
display_checks(stat_checks, "Statistical sanity checks")
PHASE0_ACCEPTANCE["statistical_sanity"] = True


,rows,open_price,close_price,year_high,year_low,avg_hourly_volume,year_return_pct
year,,,,,,,
2020,8767,7195.24,28923.63,29300.00,3782.13,2935.19,302.0
2021,8747,28923.63,46216.93,69000.00,28130.00,2917.58,59.8
2022,8760,46216.93,16542.40,48189.84,15476.00,6101.73,-64.2
2023,8759,16541.77,42283.58,44700.00,16499.01,4179.17,155.6
2024,8784,42283.58,93576.00,108353.00,38555.00,1472.12,121.3
2025,8760,93576.00,87648.22,126199.63,74508.00,880.20,-6.3
2026,2528,87648.21,74707.03,97924.49,60000.00,908.33,-14.8


Statistical sanity checks


,check,status,detail
0,all yearly highs exceed yearly lows,PASS,
1,all yearly row counts positive,PASS,
2,prices are positive,PASS,
3,volume is non-negative,PASS,


## 12. Known Anomaly Record

This is a compact summary to carry forward into Phase 1. Backtests must not assume a perfect uninterrupted hourly series.

In [14]:
anomaly_summary = {
    "start_utc": df["open_time_utc"].min(),
    "end_utc": df["open_time_utc"].max(),
    "expected_rows": len(expected_index),
    "actual_rows": len(df),
    "missing_rows": len(missing_index),
    "missing_intervals": len(gap_intervals),
    "duplicate_timestamps": int(df["open_time_utc"].duplicated().sum()),
    "zero_volume_rows": int((df["volume"] == 0).sum()),
    "sources": df["source"].value_counts(dropna=False).to_dict(),
}

display(pd.DataFrame([anomaly_summary]).T.rename(columns={0: "value"}))

if len(gap_intervals) > 0:
    documented_gaps = gap_intervals.copy()
    documented_gaps["nearby_zero_volume"] = documented_gaps.apply(
        lambda row: bool(
            ((df["volume"] == 0) &
             (df["open_time_utc"] >= row["gap_start_utc"] - pd.Timedelta(hours=6)) &
             (df["open_time_utc"] <= row["gap_end_utc"] + pd.Timedelta(hours=6))).any()
        ),
        axis=1,
    )
    documented_gaps["likely_cause"] = np.where(documented_gaps["nearby_zero_volume"], "exchange downtime / frozen market", "missing historical bar")
    display(documented_gaps)


,value
start_utc,2020-01-01 00:00:00+00:00
end_utc,2026-04-16 07:00:00+00:00
expected_rows,55136
actual_rows,55105
missing_rows,31
missing_intervals,15
duplicate_timestamps,0
zero_volume_rows,3
sources,"{'binance_vision': 54737, 'ccxt_binance': 368}"


,gap_start_utc,gap_end_utc,missing_hours,year,nearby_zero_volume,likely_cause
0,2020-02-09 02:00:00+00:00,2020-02-09 02:00:00+00:00,1,2020,False,missing historical bar
1,2020-02-19 12:00:00+00:00,2020-02-19 16:00:00+00:00,5,2020,False,missing historical bar
2,2020-03-04 10:00:00+00:00,2020-03-04 10:00:00+00:00,1,2020,False,missing historical bar
3,2020-04-25 02:00:00+00:00,2020-04-25 03:00:00+00:00,2,2020,False,missing historical bar
4,2020-06-28 02:00:00+00:00,2020-06-28 04:00:00+00:00,3,2020,False,missing historical bar
5,2020-11-30 06:00:00+00:00,2020-11-30 06:00:00+00:00,1,2020,False,missing historical bar
6,2020-12-21 15:00:00+00:00,2020-12-21 17:00:00+00:00,3,2020,True,exchange downtime / frozen market
7,2020-12-25 02:00:00+00:00,2020-12-25 02:00:00+00:00,1,2020,False,missing historical bar
8,2021-02-11 04:00:00+00:00,2021-02-11 04:00:00+00:00,1,2021,True,exchange downtime / frozen market
9,2021-03-06 02:00:00+00:00,2021-03-06 02:00:00+00:00,1,2021,False,missing historical bar


## 13. Final Phase 0 Acceptance Summary

In [15]:
acceptance_rows = [
    {"area": "canonical load", "status": pass_fail(PHASE0_ACCEPTANCE.get("canonical_load")), "evidence": "canonical parquet loads and date range is visible"},
    {"area": "schema and timestamps", "status": pass_fail(PHASE0_ACCEPTANCE.get("schema_timestamp")), "evidence": "UTC, sorted, unique, hour-aligned, required schema"},
    {"area": "source handoff", "status": pass_fail(PHASE0_ACCEPTANCE.get("source_handoff")), "evidence": "source coverage and transition boundary inspected"},
    {"area": "archive integrity", "status": pass_fail(PHASE0_ACCEPTANCE.get("archive_integrity")), "evidence": "latest archive exists and canonical has not shrunk"},
    {"area": "validation report", "status": pass_fail(PHASE0_ACCEPTANCE.get("validation_report")), "evidence": "newest validation JSON inspected"},
    {"area": "missing bar documentation", "status": pass_fail(PHASE0_ACCEPTANCE.get("missing_bar_documentation")), "evidence": "missing bars grouped and no 2024+ gaps"},
    {"area": "zero-volume documentation", "status": pass_fail(PHASE0_ACCEPTANCE.get("zero_volume_documentation")), "evidence": "zero-volume bars listed and retained"},
    {"area": "market event spot checks", "status": pass_fail(PHASE0_ACCEPTANCE.get("market_event_spot_checks")), "evidence": "known BTC events within expected ranges"},
    {"area": "statistical sanity", "status": pass_fail(PHASE0_ACCEPTANCE.get("statistical_sanity")), "evidence": "annual price/volume summary sane"},
]
acceptance_df = pd.DataFrame(acceptance_rows)
display(acceptance_df)

failed = acceptance_df[acceptance_df["status"] != "PASS"]
assert failed.empty, failed

print("PHASE 0 ACCEPTANCE PASS")
print("Canonical dataset is ready for Phase 1 backtesting, with known gaps and zero-volume bars documented.")


,area,status,evidence
0,canonical load,PASS,canonical parquet loads and date range is visible
1,schema and timestamps,PASS,"UTC, sorted, unique, hour-aligned, required schema"
2,source handoff,PASS,source coverage and transition boundary inspected
3,archive integrity,PASS,latest archive exists and canonical has not shrunk
4,validation report,PASS,newest validation JSON inspected
5,missing bar documentation,PASS,missing bars grouped and no 2024+ gaps
6,zero-volume documentation,PASS,zero-volume bars listed and retained
7,market event spot checks,PASS,known BTC events within expected ranges
8,statistical sanity,PASS,annual price/volume summary sane


PHASE 0 ACCEPTANCE PASS
Canonical dataset is ready for Phase 1 backtesting, with known gaps and zero-volume bars documented.
